In [1]:
#| default_exp attic

# Attic

A step that was part of the pipeline and is not any more. It is kept here, exported and tested, so
that a later decision to bring it back is a matter of calling it again rather than rewriting it.

**Nothing in `randseq.core` calls this.** `find_restricted_motifs` goes from step 4 straight to
reporting.

## Why step 5 was retired

Rescoring the survivors on the plasmids that carry exactly one of them changes no call anywhere in
the benchmark: the published Fig 3 motifs, the strains the paper reports nothing for, and the
shipped panel are identical with and without it. All it moves is the denominator a surviving motif
is reported over, which made `num_sequences` smaller without making any answer different.

In [2]:
#| export
import ast
import numpy as np
import pandas as pd

from randseq.coding import encode_library, motif_presence_matrix
from randseq.utils import get_lib_seq_context, _get_filter_for_motif
from randseq.core import fixed_motif_presence, fixed_motif_labels

In [3]:
#| hide
from fastcore.test import *
import pandas as pd

## From a presence matrix to a score per motif

A boolean matrix with one row per sequence and one column per candidate says which plasmids carry
which motif. A plasmid carrying two candidates cannot say which one depleted it, so it is excluded
from both; sequences whose row sums to exactly **one** are the evidence that distinguishes
candidates, and `_unique_hit_stats` reduces the matrix over those rows to three numbers per
motif.

In [4]:
#| export
def _unique_hit_stats(matrix, fc_values, log2fc_thr):
    """Per-motif statistics over sequences carrying exactly one of the motifs in `matrix`.

    Co-occurring motifs are the reason this step exists: a sequence carrying two candidates
    cannot tell us which one depleted it, so it is excluded from both. Counting how many
    candidates each sequence carries, once, avoids intersecting every pair of motifs.
    """
    fc = np.asarray(fc_values, dtype=float)
    only_one = matrix.sum(axis=1) == 1
    unique = matrix & only_one[:, None]
    n_seqs = unique.sum(axis=0)
    with np.errstate(invalid="ignore", divide="ignore"):
        n_depleted = (unique & (fc < log2fc_thr)[:, None]).sum(axis=0)
        fc_sum = (unique * fc[:, None]).sum(axis=0)
        frac = np.where(n_seqs > 0, n_depleted / np.maximum(n_seqs, 1), 0.0)
        # 0.0 rather than NaN for a motif with no unique sequences: NaN would propagate
        # through every comparison downstream, and such motifs fail the caller's filter anyway.
        avg = np.where(n_seqs > 0, fc_sum / np.maximum(n_seqs, 1), 0.0)
    return n_seqs, frac, avg

In [5]:
#| hide
# row 1 carries BOTH motifs, so it counts for neither: that is the whole point of the step
_m = np.array([[True, False], [True, True], [False, True]])
_n, _frac, _avg = _unique_hit_stats(_m, np.array([-2.0, -2.0, 0.0]), -1)
test_eq(_n.tolist(), [1, 1])
test_eq(_frac.tolist(), [1.0, 0.0])
test_eq(_avg.tolist(), [-2.0, 0.0])
# a motif left with no unique sequence gets 0, not NaN, and is dropped by the caller's filter
_n2, _f2, _a2 = _unique_hit_stats(np.array([[True, True]]), np.array([-5.0]), -1)
test_eq(_n2.tolist(), [0, 0])
test_eq(_f2.tolist(), [0.0, 0.0])

## Step 5: rescore on plasmids carrying exactly one candidate

A plasmid carrying two candidate motifs cannot tell us which one depleted it, so it is excluded
from both. What remains — plasmids carrying exactly one — is the evidence that actually
distinguishes candidates, and `num_sequences` counts those.

### `num_sequences` is a property of the candidate set, not of the motif

This is the single most important thing to understand about the output. Admit one more
overlapping candidate and every motif it overlaps loses plasmids. On the shipped panel, run
with the full pattern set:

| strain | motif | contains | counted | retained |
|---|---|---|---|---|
| EDL933 | `CACNNNNNNNCTGG` | 277 | 277 | **100.0%** |
| JJ1886 | `AAAGNNNNGTT` | 913 | 882 | **96.6%** |
| JJ1886 | `ATACNNNNGTG` | 504 | 483 | **95.8%** |
| JJ1886 | `CACNNNNGTAC` | 468 | 441 | **94.2%** |
| LMR_503 | `CACNNNNGTAC` | 468 | 431 | **92.1%** |

Retention is 92–100% everywhere on the panel, so `num_sequences` measures evidence and is stable.
**That is a result of step 4, not a property of the method.** Without it, the same LMR_503 motif
is scored on 6 of 468 plasmids — 1.3% — because nineteen shadow descriptions of the same site each
take their share. An `n=6` that is 6-of-468 means something quite different from an `n=277` that
is 277-of-277, and the two should never be compared as though they were the same quantity.

So it is worth checking retention rather than assuming it. A motif whose `num_sequences` is a
small fraction of the plasmids that contain it is telling you the candidate set is crowded, not
that the site is weak.

### The rescoring is not iterated

Dropping a motif frees its plasmids to count as unique for the survivors, so in principle the
step could be repeated to a fixed point.

It makes no difference: on every panel strain, passes 2 and 3 return exactly the numbers of
pass 1. Because step 4 has already removed the shadows, there is no crowd of co-occurring
candidates left for a second pass to release, and the first pass **is** the fixed point. One pass
is the intended semantics, and here it is stable rather than a judgement call.

In [6]:
#| export
def update_motif_scores_from_unique_hits(
    flexible_motifs_df,
    log2fc_series,
    left_context,
    right_context,
    flexible_motif_log2fc_thr=-1,
    flexible_motif_min_support=3,
    flexible_motif_score_thr=0.85
):
    """
    Updates a DataFrame of motifs with new scores based on unique sequence
    hits (sequences not matched by any other motif) and filters the result.

    Args:
        flexible_motifs_df (pd.DataFrame): DataFrame with a 'motif' column.
        log2fc_series (pd.Series): Series with DNA sequences as index and values.
        left_context (str): DNA string to prepend for matching.
        right_context (str): DNA string to append for matching.
        flexible_motif_log2fc_thr (float): The log2fc threshold for the score function.
        flexible_motif_min_support (int): The minimum number of unique sequences required.
        flexible_motif_score_thr (float): The minimum fraction_depleted score required.

    Returns:
        pd.DataFrame: A new, filtered DataFrame with updated scores.
    """
    if flexible_motifs_df.empty:
        # No candidates is a normal outcome, not an error -- an RM-deficient strain
        # reaches here legitimately. Return the empty frame with its columns intact.
        return flexible_motifs_df.copy()

    all_motifs = flexible_motifs_df['motif'].tolist()

    # Which pattern each motif came from, so presence can be read off the window codes.
    if 'pattern' in flexible_motifs_df.columns:
        patterns = []
        for p in flexible_motifs_df['pattern']:
            try:
                patterns.append(tuple(ast.literal_eval(p)) if isinstance(p, str) else tuple(p))
            except (ValueError, SyntaxError):
                patterns.append(None)
    else:
        patterns = [None] * len(all_motifs)

    encoded = encode_library(
        get_lib_seq_context(log2fc_series.index.tolist(), left_context, right_context))
    presence, handled = motif_presence_matrix(encoded, all_motifs, patterns)

    # Anything the fast path could not encode -- an IUPAC-degenerate motif, or one whose
    # pattern is unknown -- falls back to the regex search, so the answer is unchanged.
    fallback = np.flatnonzero(~handled)
    if len(fallback):
        print(f"  {len(fallback)} motif(s) need the regex path (IUPAC or unknown pattern)")
        for j in tqdm(fallback, desc="  Building filters"):
            presence[:, j] = _get_filter_for_motif(
                log2fc_series, all_motifs[j], left_context, right_context).to_numpy()

    n_seqs, frac_depleted, avg_log2fc = _unique_hit_stats(
        presence, log2fc_series.to_numpy(), flexible_motif_log2fc_thr)
    new_metrics = [{'fraction_depleted': f, 'num_sequences': int(n), 'avg_log2fc': a}
                   for f, n, a in zip(frac_depleted, n_seqs, avg_log2fc)]

    df_updated = flexible_motifs_df.copy()
    metrics_df = pd.DataFrame(new_metrics, index=df_updated.index)

    df_updated['fraction_depleted'] = metrics_df['fraction_depleted']
    df_updated['num_sequences'] = metrics_df['num_sequences']
    df_updated['avg_log2fc'] = metrics_df['avg_log2fc']

    final_df = df_updated[
        (df_updated['avg_log2fc'] < flexible_motif_log2fc_thr) &
        (df_updated['num_sequences'] >= flexible_motif_min_support) &
        (df_updated['fraction_depleted'] >= flexible_motif_score_thr)
    ]

    return final_df

In [7]:
#| hide
# Two motifs; the middle plasmid carries both and so counts for neither.
_L, _R = "AAAA", "TTTT"
_seqs = ["GGTCTCGGGGGG", "GGTCTCAAAAAA", "CCCCCCAAAAAA"]
_fc = pd.Series([-5.0, -5.0, -0.1], index=_seqs)
_df = pd.DataFrame({"motif": ["GGTCTC", "AAAAAA"], "pattern": ["(6, 0, 0)", "(6, 0, 0)"],
                    "fraction_depleted": [1.0, 0.5], "num_sequences": [2, 2],
                    "avg_log2fc": [-5.0, -2.55]})
_out = update_motif_scores_from_unique_hits(_df, _fc, _L, _R, flexible_motif_log2fc_thr=-1,
                                            flexible_motif_min_support=1,
                                            flexible_motif_score_thr=0.5)
# GGTCTC keeps its one unique plasmid and passes; AAAAAA's one unique plasmid is undepleted
test_eq(list(_out.motif), ["GGTCTC"])
test_eq(int(_out.num_sequences.iloc[0]), 1)
# an empty input is a normal outcome (a strain with no system) and keeps its columns
test_eq(list(update_motif_scores_from_unique_hits(_df.iloc[:0], _fc, _L, _R).columns),
        list(_df.columns))